In [ ]:
import torch
torch.__version__
import triton
import triton.language as tl

In [ ]:
import torch.nn.functional as F

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2])
b = torch.tensor([0.0])

z = x1 * w1 + b
a = torch.sigmoid(z)

loss = F.binary_cross_entropy(a,y)
print(loss)

In [ ]:
y.shape

In [ ]:
import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

z = x1 * w1 + b
a = torch.sigmoid(z)

loss = F.binary_cross_entropy(a,y)

grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

In [ ]:
print(grad_L_w1)
print(grad_L_w1)

In [ ]:
loss.backward()

In [ ]:
print(w1.grad)
print(b.grad)
print(x1.grad)

In [ ]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(
            # 1st hidden layer
            torch.nn.Linear(num_inputs,30),
            torch.nn.ReLU(),
            # 2nd
            torch.nn.Linear(30,20),
            torch.nn.ReLU(),
            #ouptut
            torch.nn.Linear(20,num_outputs),
        )
    def forward(self,x):
        logits = self.layers(x)
        return logits

In [ ]:
torch.manual_seed(123)
model = NeuralNetwork(50,3)

In [ ]:
print(model)

In [ ]:
print([p for p in model.parameters()])

In [ ]:
print(model.layers[0].weight.shape)
print(model.layers[2].weight.shape)
print(model.layers[4].weight.shape)

In [ ]:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(num_params) # 30*50 + 20*30 + 3 * 20 + 30 + 20 + 3

In [ ]:
torch.manual_seed(123)

X = torch.rand((1,50))
with torch.no_grad():
    out = torch.softmax(model(X), dim=1)
print(out)

In [ ]:
X.shape

In [ ]:
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])

y_train = torch.tensor([0, 0, 0, 1, 1])

X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6],
])

y_test = torch.tensor([0, 1])

In [ ]:
from torch.utils.data import Dataset


class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y

    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y

    def __len__(self):
        return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

In [ ]:
len(train_ds)

In [ ]:
from torch.utils.data import DataLoader

torch.manual_seed(123)

train_loader = DataLoader(
    dataset = train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    drop_last = True,
)

test_loader = DataLoader(
    dataset = test_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0
)

In [ ]:
for idx, (x,y) in enumerate(train_loader):
    print(f"Batch {idx+1}: ", x,y)

In [ ]:
import torch.nn.functional as F

torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)
optimizer = torch.optim.SGD(model.parameters(),lr=0.5)

num_epochs = 3

for epoch in range(num_epochs):

    model.train()
    for batch_idx, (features,labels) in enumerate(train_loader):

        logits = model(features)
        loss = F.cross_entropy(logits,labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        ### LOGGING
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")

    model.eval()
    # Optional model evaluation

In [ ]:
model.eval()

with torch.no_grad():
    outputs = model(X_train)
torch.set_printoptions(sci_mode=False)
probas = torch.softmax(outputs, dim=1)
print(probas)

In [ ]:
predictions = torch.argmax(probas, dim=1)
print(predictions)

In [ ]:
def compute_accuracy(model, dataloader):

    model = model.eval()
    correct = 0.0
    total_examples = 0

    for idx, (features, labels) in enumerate(dataloader):

        with torch.no_grad():
            logits = model(features)

        predictions = torch.argmax(logits, dim=1)
        compare = labels == predictions
        correct += torch.sum(compare)
        total_examples += len(compare)

    return (correct / total_examples).item()

In [ ]:

torch.save(model.state_dict(), "model.pth")

In [ ]:
print(torch.cuda.is_available())

In [ ]:
t1d = torch.tensor([1.,2.,3.]).to("cuda")
t2d = torch.tensor([4.,5.,6.]).to("cuda")

print(t1d + t2d)

In [ ]:
torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)

# New: Define a device variable that defaults to a GPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# New: Transfer the model onto the GPU.
model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

num_epochs = 3

for epoch in range(num_epochs):

    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):

        # New: Transfer the data onto the GPU.
        features, labels = features.to(device), labels.to(device)    #C
        logits = model(features)
        loss = F.cross_entropy(logits, labels) # Loss function

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        ### LOGGING
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")

    model.eval()
    # Optional model evaluation

In [ ]:
torch.manual_seed(12)
a = torch.rand(size=(4,3))
a

In [ ]:
a.shape

In [ ]:
a[:,:,None].shape

In [ ]:
a_max = a.max(dim=1)[0]
a_max

In [ ]:
a_max[:, None].shape

In [ ]:
#
a - a_max[:, None]

In [ ]:
s = torch.rand(10)
s.shape

In [ ]:
s[:,None]

In [ ]:
torch.manual_seed(0)
M = 4096
N = 4096
x = torch.rand((M,N))

In [ ]:
x.shape

In [ ]:
x.max(dim=1)

In [ ]:
x.stride(0)

In [ ]:
torch.rand((4,3))

In [ ]:
k = torch.rand(4,3)

In [ ]:
m, n = k.shape
m

In [ ]:
[i for i in range(1,10)] < 5

In [ ]:
pid = np.arange(0,9)
grid_n = int(96/32)
pid_m = pid // grid_n
pid_n = pid % grid_n

In [ ]:
grid_n

In [ ]:
pid_m, pid_n

In [ ]:
@triton.jit
# tl.arange(0, 10)
tl

In [ ]:
tl.arange(0,10)

In [ ]:
import numpy as np
np.arange(0,10) % 8

In [ ]:
assert 1==1

In [ ]:
np.allclose

In [ ]:
np.arange(0,81) // 27

In [ ]:
np.arange(0,81) % 27 % 3

In [ ]:

np.arange(0,81) % 27 // 3

In [ ]:
x = torch.rand(4,)

In [ ]:
x[:, None].shape

In [ ]:
x[None, :].shape

In [ ]:
x

In [ ]:
x[:, None]

In [ ]:
x[None, :]

In [ ]:
a = np.array([[0 , 1 , 2 , 3 ],
 [4 , 5 , 6 , 7 ],
 [8 , 9 , 10, 11],
 [12, 13, 14, 15]])

In [ ]:
a.shape

In [ ]:
for ij in a.reshape(16,):
    size_gj = 4 * 2
    group_id = ij // size_gj
    off_i = group_id * 2
    print(off_i)
    size_g = min(4 - off_i, 4)
    new_ij = ij % size_gj
    print(new_ij)

In [ ]:
a // 8

In [ ]:
def swizzle(i, j, size_i=4, size_j=4, group_sz=2):